# NYC 311 Maintenance Priority Modeling

This notebook walks through a maintenance-priority model using public NYC 311/HPD housing service requests. The public data is not a private work-order system, but it is close enough to practice the same analytics pattern: clean the raw requests, engineer features, define a review label, train a model, and inspect results.

## I. Setup

The notebook imports the same data and modeling utilities used by the command-line scripts.

In [2]:
!pip install plotly

  Using cached plotly-6.8.0-py3-none-any.whl.metadata (9.0 kB)
  Using cached narwhals-2.22.1-py3-none-any.whl.metadata (15 kB)
Using cached plotly-6.8.0-py3-none-any.whl (9.9 MB)
Using cached narwhals-2.22.1-py3-none-any.whl (454 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from pathlib import Path
import html
import sys

import pandas as pd
from IPython.display import HTML, display


def find_repo_root():
    known_root = Path(r"C:\Users\JosefinoJrDeGuzman\Documents\Internship Assignment\property-maintenance-priority-model")
    candidates = [known_root, Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        package_dir = candidate / "src" / "maintenance_priority"
        if package_dir.exists() and (candidate / "scripts").exists():
            return candidate
    raise FileNotFoundError("Open the property-maintenance-priority-model folder in VS Code, or keep this notebook in that repo.")


def bar_table(title, values, color="#93c5fd", value_format="{:.3f}"):
    series = pd.Series(values).dropna()
    if series.empty:
        return HTML(f"<p><strong>{html.escape(title)}</strong>: no rows to show.</p>")

    scale = max(float(series.abs().max()), 1e-9)
    rows = []
    for label, value in series.items():
        width = min(100, max(2, abs(float(value)) / scale * 100))
        rows.append(
            f"<div style='display:flex;gap:8px;align-items:center;margin:6px 0;'>"
            f"<div style='width:230px'>{html.escape(str(label))}</div>"
            f"<div style='flex:1;background:#e5e7eb;height:16px;border-radius:4px;'>"
            f"<div style='width:{width:.1f}%;background:{color};height:16px;border-radius:4px;'></div></div>"
            f"<div style='width:80px;text-align:right'>{value_format.format(float(value))}</div></div>"
        )
    return HTML(f"<h3>{html.escape(title)}</h3>" + "".join(rows))


ROOT = find_repo_root()
for folder in ("scripts", "src"):
    path = str(ROOT / folder)
    if path not in sys.path:
        sys.path.insert(0, path)

from download_data import build_training_table, download_public_requests
from maintenance_priority import LogisticPriorityModel, binary_metrics, build_feature_matrix
from maintenance_priority.inference import load_model
from explainability import contribution_summary, score_dataframe
from tables import add_aging_bucket, add_open_flag, build_triage_table, first_existing_column, style_executive_table
from visuals import (
    plot_aging_buckets,
    plot_confusion_matrix,
    plot_feature_contributions,
    plot_group_count,
    plot_issue_breakdown,
    plot_metrics_summary,
    plot_precision_recall_curve,
    plot_priority_ranking,
    plot_roc_curve,
    plot_status_summary,
)

ROOT


WindowsPath('C:/Users/JosefinoJrDeGuzman/Documents/Internship Assignment/property-maintenance-priority-model')

## II. Download and prepare Public Service Requests

The processed table contains public HPD complaint records from NYC 311. The target is a priority-follow-up proxy based on open status, closure time, and complaint category.

In [4]:
processed_path = ROOT / 'data' / 'processed' / 'nyc_311_hpd_priority_training.csv'

if processed_path.exists():
    requests_df = pd.read_csv(processed_path)
else:
    raw_requests = download_public_requests()
    requests_df = build_training_table(raw_requests)
    processed_path.parent.mkdir(parents=True, exist_ok=True)
    requests_df.to_csv(processed_path, index=False)

requests_df.head()

,unique_key,created_month,is_winter,complaint_type,descriptor,has_descriptor,borough,incident_zip,status,location_type,open_data_channel_type,closure_days,priority_followup
0,66755054,11,1,APPLIANCE,REFRIGERATOR,1,MANHATTAN,10029.0,Closed,RESIDENTIAL BUILDING,PHONE,125.635000,1
1,66778902,11,1,GENERAL,COOKING GAS,1,MANHATTAN,10016.0,Closed,RESIDENTIAL BUILDING,PHONE,124.378414,1
2,66809809,11,1,ELECTRIC,OUTLET/SWITCH,1,MANHATTAN,10024.0,Closed,RESIDENTIAL BUILDING,PHONE,120.717581,1
3,67045411,12,1,UNSANITARY CONDITION,MOLD,1,BROOKLYN,11206.0,Closed,RESIDENTIAL BUILDING,PHONE,99.882662,1
4,67175899,12,1,ELECTRIC,NO LIGHTING,1,BRONX,10460.0,Closed,RESIDENTIAL BUILDING,PHONE,89.226597,1


## III. Summary measures

I check complaint volumes, class balance, and closure-time behavior before training. This gives me a practical sense of the data quality.

In [5]:
summary = {
    'rows': len(requests_df),
    'complaint_types': requests_df['complaint_type'].nunique(),
    'boroughs': requests_df['borough'].nunique(),
    'priority_rate': requests_df['priority_followup'].mean(),
    'median_closure_days': requests_df['closure_days'].median(),
}

pd.Series(summary).to_frame('value')

,value
rows,50000.000000
complaint_types,13.000000
boroughs,5.000000
priority_rate,0.720780
median_closure_days,6.285428


In [6]:
requests_df['complaint_type'].value_counts().head(12).to_frame('request_count')

,request_count
complaint_type,
HEAT/HOT WATER,16223
UNSANITARY CONDITION,8073
PLUMBING,5299
PAINT/PLASTER,4306
DOOR/WINDOW,3517
WATER LEAK,2958
GENERAL,2679
ELECTRIC,2134
FLOORING/STAIRS,2040


## III-A. Dashboard-style data visualization

These cells mirror the new Streamlit dashboard using the reusable modules in `src/visuals.py`, `src/tables.py`, and `src/explainability.py`. The public NYC 311 dataset does not include internal property, building, or unit fields, so the grouping logic falls back to available fields such as borough, ZIP, and location type.


In [7]:
dashboard_model_path = ROOT / "artifacts" / "maintenance_priority_model.json"
dashboard_payload = load_model(dashboard_model_path)

dashboard_df, scoring_warnings = score_dataframe(requests_df, dashboard_payload, threshold=0.50)
dashboard_df, open_note = add_open_flag(
    dashboard_df,
    status_col="status" if "status" in dashboard_df.columns else None,
)
dashboard_df, age_col, age_note = add_aging_bucket(dashboard_df)

property_group_col = first_existing_column(
    dashboard_df,
    [
        "property",
        "property_id",
        "property_name",
        "building",
        "building_id",
        "building_name",
        "unit",
        "unit_id",
        "unit_number",
    ],
)
fallback_group_col = first_existing_column(dashboard_df, ["borough", "incident_zip", "location_type"])
dashboard_group_col = property_group_col or fallback_group_col
dashboard_category_col = first_existing_column(
    dashboard_df,
    ["complaint_type", "category", "issue_category", "request_type", "problem_type"],
)
dashboard_status_col = first_existing_column(dashboard_df, ["status", "work_order_status", "order_status"])
open_dashboard_df = dashboard_df[dashboard_df["_is_open"]].copy()

dashboard_notes = [note for note in [*scoring_warnings, open_note, age_note] if note]
if dashboard_notes:
    display(pd.Series(dashboard_notes, name="dashboard_note").to_frame())

pd.Series(
    {
        "rows": len(dashboard_df),
        "open_backlog_rows": int(dashboard_df["_is_open"].sum()),
        "urgent_review_rows": int((dashboard_df["priority_score"] >= 0.50).sum()),
        "grouping_column": dashboard_group_col,
        "category_column": dashboard_category_col,
        "status_column": dashboard_status_col,
    }
).to_frame("value")


,dashboard_note
0,Aging buckets use closure_days because created...


,value
rows,50000
open_backlog_rows,3141
urgent_review_rows,40924
grouping_column,borough
category_column,complaint_type
status_column,status


In [8]:
plot_priority_ranking(
    dashboard_df,
    score_col="priority_score",
    label_columns=["unique_key", "complaint_type", "borough", "incident_zip"],
    top_n=20,
)


In [9]:
group_label = dashboard_group_col.replace("_", " ").title() if dashboard_group_col else "Available Field"
plot_group_count(
    open_dashboard_df,
    dashboard_group_col,
    f"Open Work Orders By {group_label}",
    top_n=15,
)


In [10]:
display(plot_aging_buckets(dashboard_df))
display(plot_issue_breakdown(dashboard_df, dashboard_category_col))
display(plot_status_summary(dashboard_df, dashboard_status_col))


In [11]:
executive_triage_table = build_triage_table(
    dashboard_df,
    score_col="priority_score",
    threshold=0.50,
    top_n=25,
)
style_executive_table(executive_triage_table)


Rank,Request ID,Borough,ZIP,Location Type,Issue Category,Issue Detail,Status,Aging Bucket,Age Days,Priority Score,Review Label,Training Target,Recommended Action
1,67613892,BROOKLYN,11226.000000,RESIDENTIAL BUILDING,UNSANITARY CONDITION,PESTS,Closed,15+ days,57.1,95.5%,urgent_review,1,No backlog action
2,67787587,BROOKLYN,11233.000000,RESIDENTIAL BUILDING,UNSANITARY CONDITION,PESTS,Closed,15+ days,18.3,95.5%,urgent_review,1,No backlog action
3,67783622,BROOKLYN,11233.000000,RESIDENTIAL BUILDING,UNSANITARY CONDITION,SEWAGE,Closed,15+ days,17.7,95.5%,urgent_review,1,No backlog action
4,67783610,BROOKLYN,11226.000000,RESIDENTIAL BUILDING,UNSANITARY CONDITION,PESTS,Closed,15+ days,18.2,95.5%,urgent_review,1,No backlog action
5,67642022,BROOKLYN,11236.000000,RESIDENTIAL BUILDING,UNSANITARY CONDITION,PESTS,Closed,15+ days,34.0,95.5%,urgent_review,1,No backlog action
6,67438266,BROOKLYN,11207.000000,RESIDENTIAL BUILDING,UNSANITARY CONDITION,GARBAGE/RECYCLING STORAGE,Closed,15+ days,33.6,95.5%,urgent_review,1,No backlog action
7,67388055,BROOKLYN,11207.000000,RESIDENTIAL BUILDING,UNSANITARY CONDITION,MOLD,Closed,15+ days,37.7,95.5%,urgent_review,1,No backlog action
8,67389097,BROOKLYN,11221.000000,RESIDENTIAL BUILDING,UNSANITARY CONDITION,GARBAGE/RECYCLING STORAGE,Closed,15+ days,79.1,95.5%,urgent_review,1,No backlog action
9,67411831,BROOKLYN,11207.000000,RESIDENTIAL BUILDING,UNSANITARY CONDITION,PESTS,Closed,15+ days,27.2,95.5%,urgent_review,1,No backlog action
10,67406248,BROOKLYN,11207.000000,RESIDENTIAL BUILDING,UNSANITARY CONDITION,MOLD,Closed,15+ days,45.9,95.5%,urgent_review,1,No backlog action


In [12]:
recommended_actions = executive_triage_table[
    [
        column
        for column in [
            "Rank",
            "Request ID",
            "Issue Category",
            "Status",
            "Aging Bucket",
            "Priority Score",
            "Recommended Action",
        ]
        if column in executive_triage_table.columns
    ]
].head(15)
style_executive_table(recommended_actions)


Rank,Request ID,Issue Category,Status,Aging Bucket,Priority Score,Recommended Action
1,67613892,UNSANITARY CONDITION,Closed,15+ days,95.5%,No backlog action
2,67787587,UNSANITARY CONDITION,Closed,15+ days,95.5%,No backlog action
3,67783622,UNSANITARY CONDITION,Closed,15+ days,95.5%,No backlog action
4,67783610,UNSANITARY CONDITION,Closed,15+ days,95.5%,No backlog action
5,67642022,UNSANITARY CONDITION,Closed,15+ days,95.5%,No backlog action
6,67438266,UNSANITARY CONDITION,Closed,15+ days,95.5%,No backlog action
7,67388055,UNSANITARY CONDITION,Closed,15+ days,95.5%,No backlog action
8,67389097,UNSANITARY CONDITION,Closed,15+ days,95.5%,No backlog action
9,67411831,UNSANITARY CONDITION,Closed,15+ days,95.5%,No backlog action
10,67406248,UNSANITARY CONDITION,Closed,15+ days,95.5%,No backlog action


## IV. Data wrangling and feature engineering

The feature matrix one-hot encodes complaint type, borough, and reporting channel. It also keeps simple operational fields such as month, winter flag, and descriptor availability.

In [13]:
features = build_feature_matrix(requests_df)
target = requests_df['priority_followup'].astype(int)

features.head()

,created_month,is_winter,has_descriptor,is_residential_location,complaint_appliance,complaint_door_window,complaint_electric,complaint_flooring_stairs,complaint_general,complaint_heat_hot_water,...,complaint_water_leak,borough_bronx,borough_brooklyn,borough_manhattan,borough_queens,borough_staten_island,channel_mobile,channel_online,channel_phone,channel_unknown
0,11,1,1,1,1,0,0,0,0,0,...,0,0,0,1,0,0,0,0,1,0
1,11,1,1,1,0,0,0,0,1,0,...,0,0,0,1,0,0,0,0,1,0
2,11,1,1,1,0,0,1,0,0,0,...,0,0,0,1,0,0,0,0,1,0
3,12,1,1,1,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,1,0
4,12,1,1,1,0,0,1,0,0,0,...,0,1,0,0,0,0,0,0,1,0


In [14]:
feature_summary = features.agg(['mean', 'std']).T.sort_values('mean', ascending=False)
feature_summary.head(15).round(3)

,mean,std
created_month,4.179,2.832
has_descriptor,1.000,0.000
is_residential_location,1.000,0.000
is_winter,0.602,0.489
channel_online,0.480,0.500
channel_phone,0.457,0.498
borough_bronx,0.354,0.478
complaint_heat_hot_water,0.324,0.468
borough_brooklyn,0.269,0.444
borough_manhattan,0.235,0.424


## V. Correlation review

Correlation is a quick screening tool here. I use it to see which engineered fields move with the priority-follow-up proxy.

In [15]:
corr_frame = features.copy()
corr_frame['priority_followup'] = target
target_corr = corr_frame.corr(numeric_only=True)['priority_followup'].sort_values(ascending=False)
target_corr.head(15).round(3)

priority_followup                 1.000
complaint_unsanitary_condition    0.224
channel_phone                     0.157
complaint_plumbing                0.128
complaint_water_leak              0.123
complaint_electric                0.072
complaint_general                 0.046
complaint_appliance               0.039
complaint_door_window             0.034
borough_brooklyn                  0.030
complaint_flooring_stairs         0.024
created_month                     0.022
borough_queens                    0.019
is_winter                         0.008
borough_staten_island            -0.004
Name: priority_followup, dtype: float64

In [16]:
plot_corr = target_corr.drop('priority_followup').head(12).sort_values()
bar_table('Feature correlation with priority-follow-up proxy', plot_corr, color='#60a5fa')


## VI. Train/test split and logistic model

The model is a transparent logistic regression trained from scratch. I use the same deterministic split as the script so results are repeatable.

In [17]:
test_mask = (requests_df.index % 4) == 0

X_train = features.loc[~test_mask].to_numpy()
X_test = features.loc[test_mask].to_numpy()
y_train = target.loc[~test_mask].to_numpy()
y_test = target.loc[test_mask].to_numpy()

model = LogisticPriorityModel()
model.fit(X_train, y_train)

preds = model.predict(X_test)
metrics = binary_metrics(y_test, preds)
metrics

{'accuracy': 0.7666,
 'precision': 0.799,
 'recall': 0.905,
 'f1': 0.8487,
 'tp': 8179,
 'tn': 1404,
 'fp': 2058,
 'fn': 859}

### Visual model diagnostics

The dashboard uses the same saved metric fields for an executive metric summary and confusion matrix. When target labels and probabilities are present, it can also show ROC and precision-recall diagnostics.


In [18]:
plot_metrics_summary(metrics)


In [19]:
plot_confusion_matrix(metrics)


In [20]:
test_probabilities = pd.Series(model.predict_proba(X_test), index=target.loc[test_mask].index, name="priority_score")
display(plot_roc_curve(y_test, test_probabilities))
display(plot_precision_recall_curve(y_test, test_probabilities))


## VII. Model interpretation

Because the model is linear after scaling, the weights give a straightforward way to inspect what is pushing a request toward or away from priority review.

In [21]:
weights = (
    pd.DataFrame({'feature': features.columns, 'weight': model.weights_})
    .assign(abs_weight=lambda frame: frame['weight'].abs())
    .sort_values('abs_weight', ascending=False)
)

weights[['feature', 'weight']].head(15)

,feature,weight
9,complaint_heat_hot_water,-0.587943
12,complaint_unsanitary_condition,0.513652
13,complaint_water_leak,0.289363
11,complaint_plumbing,0.223339
1,is_winter,0.208111
10,complaint_paint_plaster,-0.144575
6,complaint_electric,0.133435
0,created_month,-0.073638
19,channel_mobile,-0.072292
21,channel_phone,0.061539


In [22]:
top_weights = weights.head(12).sort_values('weight')
bar_table('Top maintenance-priority model weights', top_weights.set_index('feature')['weight'], color='#a78bfa')


### Dashboard-style model explanation

The Streamlit dashboard summarizes model drivers as average absolute linear contributions. This keeps explanation close to the transparent logistic-regression artifact without inventing new business logic.


In [23]:
trained_payload = model.to_dict(features.columns, metrics)
model_contrib_df, contribution_warnings = contribution_summary(requests_df, trained_payload, limit=15)
if contribution_warnings:
    display(pd.Series(contribution_warnings, name="contribution_note").to_frame())
model_contrib_df[["business_label", "mean_contribution", "mean_abs_contribution", "direction"]].head(10)


,business_label,mean_contribution,mean_abs_contribution,direction
9,Complaint: Heat Hot Water,-0.000544,0.550518,Lowers score
12,Complaint: Unsanitary Condition,-0.001569,0.378005,Lowers score
1,Winter Request,0.000538,0.203725,Raises score
11,Complaint: Plumbing,-0.000785,0.137495,Lowers score
13,Complaint: Water Leak,0.001071,0.136539,Raises score
10,Complaint: Paint Plaster,-0.000867,0.081123,Lowers score
21,Channel: Phone,0.000118,0.061307,Raises score
0,Created Month,0.000274,0.056254,Raises score
6,Complaint: Electric,-0.000026,0.053944,Lowers score
14,Borough: Bronx,-0.000052,0.047976,Lowers score


In [24]:
plot_feature_contributions(model_contrib_df, title="Model Driver Summary")


## VIII. Prediction distribution

The probability distribution helps tune the threshold later. A threshold should be selected with operations input, not just by accepting the default `0.50`.

In [25]:
probabilities = pd.Series(model.predict_proba(X_test), name='probability')
bands = pd.cut(probabilities, bins=10).value_counts().sort_index()
bar_table('Predicted priority-follow-up probability', bands, color='#f59e0b', value_format='{:.0f}')


## IX. Save a model payload

The command-line training script writes this JSON payload to `artifacts/maintenance_priority_model.json`.

In [26]:
payload = model.to_dict(features.columns, metrics)
payload.keys()

dict_keys(['model_type', 'feature_names', 'weights', 'bias', 'mean', 'scale', 'metrics', 'privacy_note'])